# AI Agent: 실시간 웹 검색 에이전트 (ReAct:Reasoning + Acting)

## AI 에이전트(AI Agent)란?
단순한 LLM(ChatGPT 등)은 묻는 말에 대답만 할 수 있지만, **에이전트(Agent)**는 스스로 계획을 세우고 **도구(Tool)**를 사용하여 실제 행동을 합니다.

- 에이전트의 핵심 작동 원리:
에이전트는 ReAct (Reasoning + Acting) 방식을 주로 사용합니다.

  - Reasoning (생각/추론): "사용자가 현재 대통령을 물어봤네? 내 학습 데이터는 옛날 거니까 검색 도구를 써야겠다."
  - Acting (행동/도구사용): 실제로 검색(Search) 도구를 실행합니다.
  - Observation (관찰): 검색 결과를 읽어봅니다. "아, 결과에 따르면 현재 대통령은 OOO이구나."
  - Final Answer (최종답변): "현재 한국의 대통령은 OOO입니다."라고 사용자에게 대답합니다.

이 개념을 바탕으로, LangGraph가 자동으로 만들어주는 에이전트를 실습

## ReAct 에이전트란?
**ReAct (Reasoning + Acting)**는 AI가 스스로 "생각(Reasoning)"하고, 필요한 "행동(Acting, 도구 사용)"을 결정하는 패턴입니다.

- LangGraph 맛보기 실습: 노드와 조건부 엣지를 우리가 직접 연결했습니다. (수동)

- 이번 실습: create_react_agent 함수를 사용하여 단 한 줄로 에이전트를 만듭니다. (자동)

이 에이전트는 사용자의 질문에 답하기 위해 인터넷 검색이 필요한지 스스로 판단하고 검색합니다.

In [ ]:
"""
AI 에이전트(AI Agent) 실습: 실시간 웹 검색 에이전트 (ReAct: Reasoning + Acting)

이번 실습에서는 LangGraph의 고수준 API를 사용하여 ReAct 에이전트를 빠르게 구축합니다.
ReAct 에이전트는 LLM이 스스로 "생각(Reasoning)"하고 "행동(Acting, 도구사용)"을 결정합니다.
"""

# =============================================
# 1단계: 필수 라이브러리 설치
# =============================================
# 이번 실습에서 사용할 주요 라이브러리들:
# - langgraph: 에이전트 그래프를 구축하기 위한 핵심 라이브러리
# - langchain-openai: OpenAI 모델 통합
# - langchain-community: 커뮤니티 제공 도구들
# - duckduckgo-search: 무료 웹 검색 API (DuckDuckGo)
# - ddgs: DuckDuckGo 검색을 위한 Python 클라이언트

# !는 Colab에서 셀 명령어 실행을 의미합니다
!pip install -qU langgraph langchain-openai langchain-community duckduckgo-search ddgs

In [ ]:
import os
import getpass

# =============================================
# 2단계: OpenAI API 키 설정
# =============================================
# LLM 모델(gpt-4o-mini)을 사용하기 위해 API 키가 필요합니다.
# 환경 변수에 키가 없으면 사용자에게 안전하게 입력받습니다.

# os.environ: Python의 환경 변수 딕셔너리
# OPENAI_API_KEY 환경 변수가 설정되어 있지 않다면
if "OPENAI_API_KEY" not in os.environ:
    # getpass.getpass(): 사용자 입력을 안전하게 받음 (터미널에 표시 안됨)
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key 입력: ")

# =============================================
# AI 에이전트(AI Agent) 개념 설명
# =============================================
"""
AI 에이전트란 단순한 챗봇을 넘어서서 스스로 계획을 세우고 도구를 사용하여
실제 행동을 수행하는 지능형 시스템입니다.

에이전트의 핵심 작동 원리: ReAct (Reasoning + Acting) 패턴
1. Reasoning (생각/추론):
   - "사용자가 현재 대통령을 물어봤네? 내 학습 데이터는 옛날 거니까 검색 도구를 써야겠다."
   - LLM이 현재 상황을 분석하고 필요한 행동을 계획

2. Acting (행동/도구사용):
   - 실제로 검색(Search) 도구를 실행
   - 외부 세계와 상호작용하여 정보 획득

3. Observation (관찰):
   - 검색 결과를 읽고 분석: "아, 결과에 따르면 현재 대통령은 OOO이구나."
   - 도구 실행 결과를 관찰하고 이해

4. Final Answer (최종답변):
   - "현재 한국의 대통령은 OOO입니다."라고 사용자에게 최종 응답
   - 관찰 결과를 바탕으로 자연스러운 답변 생성

이번 실습에서는 LangGraph의 create_react_agent 함수를 사용하여
이러한 ReAct 에이전트를 단 한 줄로 생성합니다.
"""

# =============================================
# 이전 실습(LangGraph 맛보기)과의 차이점
# =============================================
"""
이전 실습(LangGraph 맛보기):
- 노드(chatbot, tools)와 조건부 엣지를 우리가 직접 설계하고 연결
- State 객체와 상태 흐름을 수동으로 관리
- 계산기 도구 하나만 사용하는 비교적 간단한 구조

이번 실습(ReAct 에이전트):
- create_react_agent 함수가 내부적으로 모든 그래프 구조를 자동 생성
- 복잡한 ReAct 로직(생각→행동→관찰→응답)이 이미 구현되어 있음
- 웹 검색 도구를 사용하여 실시간 정보 획득 가능
- 더 복잡한 의사결정과 도구 사용 패턴 지원
"""

# =============================================
# DuckDuckGo 검색 도구 설명
# =============================================
"""
DuckDuckGoSearchRun 도구는 다음과 같은 특징이 있습니다:
1. 무료 API: 별도의 API 키 없이 무료로 사용 가능
2. 개인정보 보호: 검색 기록을 추적하지 않음
3. 실시간 정보: 최신 웹 페이지 검색 가능
4. 광고 없음: 검색 결과에 광고가 포함되지 않음

이 도구를 통해 에이전트는 LLM의 학습 데이터에 없는 최신 정보를 얻을 수 있습니다.
예: 오늘 날씨, 최근 뉴스, 실시간 주가 등
"""

# =============================================
# 다음 단계에서 진행할 내용
# =============================================
"""
다음 코드에서는:
1. ChatOpenAI LLM 초기화
2. DuckDuckGoSearchRun 도구 생성
3. create_react_agent 함수를 사용한 에이전트 생성
4. 생성된 에이전트를 사용한 실시간 검색 테스트

이제 실제 에이전트를 생성하고 테스트해 보겠습니다.
"""

OpenAI API Key 입력: ··········


## 검색 도구 및 에이전트 생성
여기서 중요한 점은 그래프를 구성하는 코드가 사라지고, create_react_agent 함수 하나로 대체된다는 점입니다.

In [ ]:
# =============================================
# 3단계: 검색 도구 및 에이전트 생성
# =============================================
# 이 단계에서는 ReAct 에이전트를 구성하는 3가지 핵심 요소를 만듭니다:
# 1. LLM (생각/추론 담당)
# 2. 도구 (행동/검색 담당)
# 3. 에이전트 실행기 (ReAct 로직 통합)

from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent

# =============================================
# 1. LLM 정의 - Reasoning(생각)을 담당할 모델
# =============================================
# ChatOpenAI: OpenAI의 언어 모델을 사용하기 위한 클래스
# model="gpt-4o-mini": 경제적이면서도 충분한 성능의 모델 선택
llm = ChatOpenAI(model="gpt-4o-mini")

# 이 LLM의 역할:
# - 사용자 질문 분석: "이 질문에 답하려면 어떤 정보가 필요할까?"
# - 도구 사용 여부 결정: "검색이 필요할까, 아니면 내 지식으로 답할 수 있을까?"
# - 검색 결과 해석: "검색 결과에서 어떤 정보를 추출해야 할까?"
# - 최종 답변 구성: "얻은 정보를 어떻게 자연스럽게 전달할까?"

# =============================================
# 2. 도구(Tool) 정의 - Acting(행동)을 담당할 검색 도구
# =============================================
# DuckDuckGoSearchRun: 웹 검색을 수행하는 도구
# API Key 없이 무료로 사용 가능한 검색 엔진: https://start.duckduckgo.com/
search_tool = DuckDuckGoSearchRun()

# 이 도구의 작동 방식:
# 1. 입력: 검색 쿼리(문자열)
# 2. 처리: DuckDuckGo에 검색 요청
# 3. 출력: 검색 결과 요약 텍스트
# 4. 특징: 실시간 정보, 개인정보 보호, 광고 없는 검색 결과

# 도구 리스트 생성 (현재는 검색 도구 하나만 있지만, 여러 도구 추가 가능)
tools = [search_tool]

# 도구의 설명문 자동 생성:
# DuckDuckGoSearchRun 도구는 자동으로 다음과 같은 설명을 생성합니다:
# "현재 웹에서 정보를 검색하는 도구입니다. 특정 주제나 최신 정보가 필요할 때 사용하세요."
# 이 설명은 LLM이 언제 이 도구를 사용해야 하는지 이해하는 데 도움을 줍니다.

# =============================================
# 3. 미리 만들어진 ReAct 에이전트 생성
# =============================================
# create_react_agent: LangGraph가 제공하는 고수준 API
# 이 함수는 내부적으로 복잡한 ReAct 에이전트 그래프를 자동으로 생성합니다.
agent_executor = create_react_agent(llm, tools)

# 이 한 줄의 코드가 내부적으로 생성하는 것들:
# 1. 그래프 구조: __start__ → agent → 조건부 분기 → 도구 실행 → agent → ...
# 2. ReAct 로직: 생각(Reasoning) 단계와 행동(Acting) 단계를 반복하는 루프
# 3. 상태 관리: 메시지 히스토리를 유지하며 에이전트 상태 관리
# 4. 도구 통합: LLM의 도구 호출을 실제 도구 실행으로 변환

# agent_executor 객체의 특징:
# - 입력: {"messages": [HumanMessage(질문)]}
# - 출력: 최종 AIMessage(답변)을 포함한 메시지 시퀀스
# - 스트리밍 지원: 실행 과정을 단계별로 관찰 가능
# - 내부 그래프: 자동 생성된 LangGraph 인스턴스

print("---검색 에이전트 생성 완료!")

# =============================================
# create_react_agent가 자동으로 생성하는 내부 구조:
# =============================================
"""
create_react_agent(llm, tools)가 생성하는 내부 그래프 구조:

        +-----------+
        | __start__ |
        +-----------+
              |
              v
        +-----------+
        |   agent   |  <-- LLM이 생각(Reasoning)하고 도구 호출 결정
        +-----------+
              |
              | (조건부 분기)
              v
    +---------------------+
    | 도구 호출 있음?       |
    +---------------------+
     |                    |
     v (있음)             v (없음)
+-----------+        +-----------+
|   tools   |        |  __end__  |  <-- 최종 답변
+-----------+        +-----------+
     |                    ^
     v                    |
+-----------+             |
|   agent   |  -----------+  <-- 도구 결과를 받고 다시 생각
+-----------+

이 구조는 다음과 같은 ReAct 사이클을 구현합니다:
1. Agent(LLM)가 사용자 질문을 분석하고 생각(Reasoning)
2. 검색이 필요하다고 판단하면 search_tool 호출 결정
3. Tools 노드에서 실제 검색 실행(Acting)
4. 검색 결과를 받아 Agent(LLM)로 다시 이동(Observation)
5. LLM이 검색 결과를 분석하고 최종 답변 생성
6. 추가 검색이 필요 없으면 종료
"""

# =============================================
# 이전 수동 방식 vs 현재 자동 방식 비교:
# =============================================
"""
수동 방식 (이전 실습):
def chatbot(state): ...
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph = graph_builder.compile()

자동 방식 (현재 실습):
agent_executor = create_react_agent(llm, tools)

차이점:
1. 코드량: 수동 10줄 이상 → 자동 1줄
2. 유지보수: 직접 관리 → LangGraph가 관리
3. 기능: 기본적인 도구 호출 → 완전한 ReAct 패턴
4. 확장성: 직접 추가 구현 → 다양한 내장 기능 활용
"""

# =============================================
# 에이전트의 실행 모드:
# =============================================
"""
생성된 agent_executor는 여러 가지 방식으로 실행할 수 있습니다:

1. stream() 메서드: 실행 과정을 실시간으로 스트리밍 (디버깅에 유용)
   for chunk in agent_executor.stream(inputs, stream_mode="values"):
       print(chunk)

2. invoke() 메서드: 한 번에 모든 실행을 수행하고 최종 결과만 반환
   result = agent_executor.invoke(inputs)

3. astream() 메서드: 비동기 스트리밍 (고성능 웹 앱에 적합)
   async for chunk in agent_executor.astream(inputs):
       print(chunk)
"""

# =============================================
# 에이전트의 의사결정 프로세스:
# =============================================
"""
에이전트가 질문을 받았을 때의 내부 의사결정 과정:

질문: "2025년 11월 현재 주요 뉴스는 뭐야?"

1. 생각 단계 (Reasoning):
   LLM: "이 질문은 최신 정보를 요구한다. 나의 학습 데이터는 2023년까지다.
         따라서 웹 검색을 통해 최신 뉴스를 찾아야 한다."

2. 행동 단계 (Acting):
   LLM: search_tool 호출 → "2025년 11월 주요 뉴스"

3. 관찰 단계 (Observation):
   검색 결과: "2025년 11월 주요 뉴스: 기후 변화 정상회의, 신기술 발표..."

4. 재생각 단계 (Reasoning):
   LLM: "검색 결과를 받았다. 이 정보를 바탕으로 사용자에게 답변을 구성해야 한다.
         추가 검색이 필요할까? → 아니오, 이 정보면 충분하다."

5. 응답 단계 (Answering):
   최종 답변: "2025년 11월 현재 주요 뉴스는..."
"""

# =============================================
# 다음 단계 예고:
# =============================================
"""
다음 코드에서는 생성된 에이전트를 실제로 테스트합니다:
1. 최신 정보가 필요한 질문 (검색 도구 사용)
2. 일반 상식 질문 (검색 없이 답변 가능)

이를 통해 에이전트가 상황에 맞게 도구 사용 여부를 스스로 판단하는지 확인합니다.
"""

---검색 에이전트 생성 완료!


/tmp/ipython-input-876092235.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


## 에이전트 테스트 (실시간 검색)
LLM은 최신 정보를 모릅니다(예: 오늘 날씨, 최근 뉴스 등). 하지만 이 에이전트는 검색 도구를 사용해 답변해냅니다.

In [ ]:
# =============================================
# 4단계: 에이전트 테스트 함수 정의 및 실행
# =============================================
# 이 단계에서는 생성된 ReAct 에이전트를 테스트하기 위한 함수를 정의합니다.
# 이 함수는 에이전트의 실행 과정을 실시간으로 관찰할 수 있게 해줍니다.

from langchain_core.messages import HumanMessage

def run_agent(question):
    """
    ReAct 에이전트를 실행하고 그 과정을 단계별로 출력하는 함수

    Args:
        question (str): 사용자의 질문

    작동 원리:
    1. 에이전트에 질문 입력
    2. 스트리밍 방식으로 실행 과정 관찰
    3. 각 단계별로 "생각", "행동", "결과", "답변" 출력
    """
    print(f"\n 질문: {question}")
    print("-" * 50)  # 구분선 (가독성 향상)

    # =============================================
    # 1. 에이전트 입력 구성
    # =============================================
    # LangGraph 에이전트는 항상 messages 형식의 입력을 기대합니다.
    # HumanMessage: 사용자의 질문을 나타내는 메시지 타입
    inputs = {"messages": [HumanMessage(content=question)]}

    # inputs의 구조:
    # {
    #   "messages": [
    #       HumanMessage(content="2025년 11월 현재 주요 뉴스는 뭐야?")
    #   ]
    # }

    # =============================================
    # 2. 에이전트 스트리밍 실행
    # =============================================
    # agent_executor.stream(): 그래프 실행을 실시간으로 스트리밍
    # stream_mode="values": 각 노드 실행 후의 상태 값(State)을 반환

    for chunk in agent_executor.stream(inputs, stream_mode="values"):
        # chunk 구조: {"messages": [모든 메시지들의 리스트]}
        # 각 청크는 그래프의 한 노드가 실행된 후의 전체 상태를 나타냅니다.

        # 가장 최근에 추가된 메시지만 확인 (마지막 메시지)
        # 이렇게 하면 각 단계에서 무엇이 발생했는지 명확히 알 수 있습니다.
        message = chunk["messages"][-1]

        # =============================================
        # 3. 메시지 타입별 처리 및 출력
        # =============================================

        # AI 메시지인 경우 (LLM이 생성한 메시지)
        if message.type == "ai":
            # 도구 호출이 포함된 경우 - "생각(Reasoning)" 단계
            if message.tool_calls:
                print(f"================[생각] 검색이 필요해! -> '{message.tool_calls[0]['args']}' 검색 시도...")

                # tool_calls 구조:
                # [{
                #   'name': 'duckduckgo_search',  # 도구 이름
                #   'args': {'query': '2025년 11월 주요 뉴스'},  # 검색어
                #   'id': 'call_...'  # 고유 호출 ID
                # }]

            # 도구 호출이 없는 경우 - "최종 답변" 단계
            else:
                print(f"================[답변] {message.content}")
                # content: LLM이 생성한 최종 답변 텍스트

        # 도구 메시지인 경우 - "행동(Acting) 결과" 단계
        elif message.type == "tool":
            # ToolMessage: 도구 실행 결과를 담은 메시지 타입
            # content: 검색 결과 텍스트 (일부만 출력)
            print(f"================[결과] 검색 완료 (내용 일부: {message.content[:100]}...)")

            # [:100]: 검색 결과의 처음 100자만 출력 (너무 길면 가독성 저하)
            # 실제 검색 결과는 수천 자일 수 있으므로 요약하여 보여줌

# =============================================
# 함수의 실행 흐름 분석:
# =============================================
"""
실행 예시: run_agent("2025년 11월 현재 주요 뉴스는 뭐야?")

1. 질문 출력:
   질문: 2025년 11월 현재 주요 뉴스는 뭐야?
   --------------------------------------------------

2. 첫 번째 청크 (AI 메시지 - 생각 단계):
   [생각] 검색이 필요해! -> '{'query': '2025년 11월 주요 뉴스'}' 검색 시도...

   LLM이 "이 질문은 최신 정보가 필요하니 검색해야겠다"고 판단

3. 두 번째 청크 (Tool 메시지 - 행동 결과 단계):
   [결과] 검색 완료 (내용 일부: 2025년 11월 주요 뉴스: 글로벌 기후 정상회의가...)

   DuckDuckGo에서 검색을 실행하고 결과를 받음

4. 세 번째 청크 (AI 메시지 - 최종 답변 단계):
   [답변] 2025년 11월 현재 주요 뉴스는 다음과 같습니다: ...

   LLM이 검색 결과를 바탕으로 최종 답변 생성
"""

# =============================================
# 출력 형식의 의미:
# =============================================
"""
================[생각] ...  : LLM의 추론 과정 (Reasoning)
================[결과] ...  : 도구 실행 결과 (Observation)
================[답변] ...  : 최종 응답 (Final Answer)

이 형식을 통해 ReAct 패턴의 각 단계를 명확히 구분할 수 있습니다.
"""

# =============================================
# 스트리밍 모드의 장점:
# =============================================
"""
1. 실시간 피드백: 사용자가 에이전트의 생각 과정을 볼 수 있음
2. 디버깅 용이: 어느 단계에서 문제가 발생했는지 쉽게 파악
3. 사용자 경험: 기다리는 동안 진행 상황을 알 수 있어 답답함 감소
4. 교육적 가치: AI의 의사결정 과정을 이해하는 데 도움

반면, invoke() 메서드는 한 번에 모든 결과를 반환하므로
실행 속도는 빠르지만 과정을 볼 수 없습니다.
"""

# =============================================
# 메시지 타입 설명:
# =============================================
"""
LangGraph/LangChain에서 사용하는 주요 메시지 타입:

1. HumanMessage: 사용자가 입력한 메시지
2. AIMessage: LLM이 생성한 메시지
   - tool_calls 속성이 있으면 도구 호출을 의미
   - 없으면 일반 응답을 의미
3. ToolMessage: 도구 실행 결과 메시지
   - name: 사용된 도구 이름
   - content: 도구 실행 결과
4. SystemMessage: 시스템 설정 메시지 (이 코드에서는 사용 안 함)

이 메시지들은 State 객체의 messages 리스트에 순서대로 누적됩니다.
"""

# =============================================
# 다음 단계 예고:
# =============================================
"""
다음 코드에서는 이 함수를 사용하여 두 가지 테스트를 실행합니다:

1. 최신 정보가 필요한 질문 (검색 도구 사용):
   run_agent("2025년 11월 현재 주요 뉴스는 뭐야?")

   예상 흐름: 생각 → 결과 → 답변 (검색 발생)

2. 일반 상식 질문 (검색 없이 답변 가능):
   run_agent("사과는 영어로 뭐야?")

   예상 흐름: 생각 → 답변 (검색 없이 바로 답변)

이를 통해 에이전트가 상황에 맞게 도구 사용을 자동으로 결정하는지 확인합니다.
"""

# =============================================
# 에이전트의 의사결정 로직:
# =============================================
"""
에이전트가 도구 사용 여부를 결정하는 기준:

1. 검색이 필요한 경우:
   - 시간에 민감한 정보 (뉴스, 날씨, 주가 등)
   - LLM 학습 데이터 이후의 정보
   - 특정 웹사이트나 최신 자료가 필요한 정보
   - 구체적인 데이터나 통계가 필요한 경우

2. 검색이 필요 없는 경우:
   - 일반 상식 (사과는 apple)
   - 언어 번역
   - 학습 데이터에 포함된 역사적 사실
   - 개념 설명이나 정의

create_react_agent는 내부적으로 LLM이 이러한 판단을
스스로 할 수 있도록 프롬프트 엔지니어링이 되어 있습니다.
"""

In [ ]:
# =============================================
# 5단계: 에이전트 실전 테스트 및 결과 분석
# =============================================
# 이 단계에서는 생성된 ReAct 에이전트를 두 가지 유형의 질문으로 테스트합니다.
# 이를 통해 에이전트가 상황에 맞게 도구 사용을 스스로 결정하는지 확인합니다.

# --- 테스트 1 실행 ---
# 1. 최신 정보가 필요한 질문 (LLM 학습 데이터에 없는 내용)
# GPT-4o-mini의 학습 데이터는 2023년 10월까지이므로, 2025년 정보는 검색이 필요합니다.
run_agent("2025년 11월 현재 주요 뉴스는 뭐야?")

# 실행 결과:
# 질문: 2025년 11월 현재 주요 뉴스는 뭐야?
# --------------------------------------------------
# ================[생각] 검색이 필요해! -> '{'query': 'November 2025 current news'}' 검색 시도...
# ================[결과] 검색 완료 (내용 일부: November 2025 was the eleventh month of the current common year. The month, which began on a Saturda...)
# ================[답변] 2025년 11월의 주요 뉴스 내용은 다음과 같습니다:
#
# 1. **열대성 사이클론 센야르**: 이 강력한 열대성 사이클론과 다른 사이클론 계통들이 동시에 발생하면서 위기가 극에 달했습니다. 북동 몬순과 라니냐 조건이 경합하는 가운데 자연 재해의 영향이 심각했습니다.
#
# 2. **글로벌 갈등과 인도적 위기**: 가자 지역에서의 인도적 위기와 함께 전 세계적으로 갈등이 격화되고 있습니다. 이러한 상황은 국제 외교 및 안보 분야에서 중요한 발전을 초래하고 있습니다.
#
# 3. **미국의 정치 및 정부 상황**: 미국에서는 선거일 드라마, 정부 셧다운 협상, 그리고 대통령 인터뷰가 주요 뉴스로 보도되었습니다.
#
# 이와 같은 사건들은 2025년 11월의 보도에서 중요한 이슈로 다뤄지고 있습니다.

# =============================================
# 테스트 1 실행 과정 분석:
# =============================================
"""
실행 단계별 분석:

1. 질문 인식:
   - 사용자: "2025년 11월 현재 주요 뉴스는 뭐야?"
   - LLM 학습 데이터: 2023년 10월까지 (구식 정보)

2. 생각(Reasoning) 단계:
   - LLM 분석: "이 질문은 최신 정보가 필요하다. 나의 학습 데이터는 2023년까지다. 검색이 필요하다."
   - 도구 호출 결정: DuckDuckGoSearchRun 사용
   - 검색어 생성: 'November 2025 current news' (자동 번역/최적화)
   - 출력: "[생각] 검색이 필요해! -> '{'query': 'November 2025 current news'}' 검색 시도..."

3. 행동(Acting) 단계 (도구 실행):
   - 도구: DuckDuckGoSearchRun
   - 실행: 'November 2025 current news' 검색
   - 결과: 실제 웹 검색 결과 수신
   - 출력: "[결과] 검색 완료 (내용 일부: November 2025 was the eleventh month...)"

4. 관찰(Observation) 및 재생각 단계:
   - LLM이 검색 결과 분석
   - "이 결과에서 주요 뉴스를 추출해야겠다. 추가 검색은 필요 없을 것 같다."

5. 최종 답변 단계:
   - 검색 결과를 바탕으로 구조화된 답변 생성
   - 3가지 주요 뉴스 카테고리로 요약
   - 자연스러운 한국어로 응답
   - 출력: "[답변] 2025년 11월의 주요 뉴스 내용은 다음과 같습니다: ..."
"""


 질문: 2025년 11월 현재 주요 뉴스는 뭐야?
--------------------------------------------------
================[생각] 검색이 필요해! -> '{'query': 'November 2025 current news'}' 검색 시도...
================[결과] 검색 완료 (내용 일부: November 2025 was the eleventh month of the current common year. The month, which began on a Saturda...)
================[답변] 2025년 11월의 주요 뉴스 내용은 다음과 같습니다:

1. **열대성 사이클론 센야르**: 이 강력한 열대성 사이클론과 다른 사이클론 계통들이 동시에 발생하면서 위기가 극에 달했습니다. 북동 몬순과 라니냐 조건이 경합하는 가운데 자연 재해의 영향이 심각했습니다.

2. **글로벌 갈등과 인도적 위기**: 가자 지역에서의 인도적 위기와 함께 전 세계적으로 갈등이 격화되고 있습니다. 이러한 상황은 국제 외교 및 안보 분야에서 중요한 발전을 초래하고 있습니다.

3. **미국의 정치 및 정부 상황**: 미국에서는 선거일 드라마, 정부 셧다운 협상, 그리고 대통령 인터뷰가 주요 뉴스로 보도되었습니다. 

이와 같은 사건들은 2025년 11월의 보도에서 중요한 이슈로 다뤄지고 있습니다.


In [ ]:
# --- 테스트 2 실행 ---
# 2. 일반 상식 질문 (검색 없이 대답 가능한지 확인)
# "사과는 영어로 뭐야?"는 LLM의 기본 지식으로 답변 가능한 일반 상식입니다.
run_agent("사과는 영어로 뭐야?")

# 실행 결과:
# 질문: 사과는 영어로 뭐야?
# --------------------------------------------------
# ================[답변] 사과는 영어로 "apple"이라고 합니다.

# =============================================
# 테스트 2 실행 과정 분석:
# =============================================
"""
실행 단계별 분석:

1. 질문 인식:
   - 사용자: "사과는 영어로 뭐야?"
   - LLM 학습 데이터: 기본적인 어휘 및 번역 지식 포함

2. 생각(Reasoning) 단계:
   - LLM 분석: "이 질문은 일반 상식이다. 내 학습 데이터에 있는 정보다. 검색 없이 바로 답변할 수 있다."
   - 도구 호출 필요성: 없음 (tool_calls 속성 없음)

3. 최종 답변 단계 (검색 없이 직접 응답):
   - LLM이 내부 지식에서 "apple"이라는 답변 추출
   - 간결하고 정확한 답변 생성
   - 출력: "[답변] 사과는 영어로 "apple"이라고 합니다."

특이사항: 생각 단계가 출력되지 않음
- 이유: LLM이 도구 호출을 결정하지 않았으므로, AI 메시지에 tool_calls 속성이 없음
- 따라서 run_agent 함수에서 "생각" 메시지를 출력하는 조건(tool_calls 존재)이 충족되지 않음
- 직접 최종 답변만 출력됨
"""

# =============================================
# 두 테스트의 비교 분석:
# =============================================
"""
| 구분                  | 테스트 1 (최신 뉴스 질문)                  | 테스트 2 (일반 상식 질문)         |
|----------------------|------------------------------------------|--------------------------------|
| 질문 유형            | 시간에 민감한 최신 정보                  | 고정된 일반 상식               |
| LLM 학습 데이터      | 부족함 (2023년까지)                      | 충분함 (기본 어휘 포함)        |
| 도구 사용 여부       | 필요함 (검색 실행)                       | 필요 없음 (직접 답변)          |
| 실행 경로            | 생각 → 행동 → 관찰 → 답변 (ReAct 사이클) | 생각 → 답변 (직접 응답)        |
| 그래프 노드 실행     | agent → tools → agent (순환 발생)       | agent → __end__ (단일 실행)    |
| 처리 시간            | 더 길음 (검색 시간 포함)                 | 짧음 (즉시 응답)               |
| 정보 신뢰도          | 실시간 웹 정보 (최신성 있음)             | 학습 데이터 기반 (정확성 있음) |

핵심 교훈:
1. 에이전트는 상황에 맞게 도구 사용을 스스로 결정함
2. 불필요한 검색을 하지 않아 리소스 절약
3. ReAct 패턴이 실제로 구현되어 작동함
"""

# =============================================
# ReAct 에이전트의 실제 작동 메커니즘:
# =============================================
"""
create_react_agent가 내부적으로 생성한 그래프의 실제 실행 과정:

테스트 1 (검색 필요한 경우):
1. __start__ → agent (초기 실행)
2. agent: LLM이 "검색 필요" 판단 → tool_calls 생성
3. 조건부 분기: tool_calls 있음 → tools 노드로 이동
4. tools: DuckDuckGoSearchRun 실행 → 검색 결과 생성
5. tools → agent: 검색 결과를 가지고 다시 LLM으로
6. agent: 검색 결과 분석 → 추가 검색 필요 없음 → 최종 답변 생성
7. 조건부 분기: tool_calls 없음 → __end__ (종료)

테스트 2 (검색 필요 없는 경우):
1. __start__ → agent (초기 실행)
2. agent: LLM이 "검색 불필요" 판단 → tool_calls 없이 바로 답변
3. 조건부 분기: tool_calls 없음 → __end__ (종료)

이러한 자동화된 의사결정이 LangGraph의 강력한 기능입니다.
"""

# =============================================
# 검색 결과의 한계 및 개선점:
# =============================================
"""
테스트 1의 검색 결과 분석:
- 검색어: 'November 2025 current news' (영어로 자동 변환)
- 결과: 위키피디아 스타일의 월별 개요 페이지
- 한계: 실제 최신 뉴스 기사보다는 일반적인 정보

개선 방안:
1. 검색어 최적화: '2025년 11월 최신 뉴스 헤드라인' 등 더 구체화
2. 다중 검색: 여러 뉴스 소스에서 정보 수집
3. 결과 필터링: 뉴스 기사만 선별하도록 개선
4. 한국어 검색: '2025년 11월 주요 뉴스'로 직접 검색
"""

# =============================================
# 실제 응용 가능성:
# =============================================
"""
이 ReAct 에이전트 패턴의 실제 적용 사례:

1. 고객 서비스 봇:
   - 회사 정책 질문: 내부 지식으로 응답
   - 주문 상태 확인: DB 조회 도구 사용
   - 배송 시간 문의: 외부 배송사 API 호출

2. 연구 보조원:
   - 개념 설명: LLM 지식으로 응답
   - 최신 논문 검색: 학술 DB 검색 도구
   - 데이터 분석: 통계 계산 도구

3. 개인 비서:
   - 일정 관리: 캘린더 API 도구
   - 날씨 확인: 날씨 API 도구
   - 뉴스 요약: 웹 검색 + 요약 도구

4. 교육 도구:
   - 기본 개념: LLM이 직접 설명
   - 실생활 예시: 웹 검색으로 최신 사례 찾기
   - 문제 풀이: 계산기 도구로 수학 문제 해결
"""

# =============================================
# LangGraph ReAct 에이전트의 핵심 가치:
# =============================================
"""
1. 자동화된 의사결정:
   - 개발자가 모든 경우의 수를 코딩할 필요 없음
   - LLM이 상황에 맞는 최적의 행동 선택

2. 모듈화와 재사용성:
   - 도구만 교체하면 다양한 에이전트 생성 가능
   - 검색, 계산, DB 조회 등 다양한 도구 통합

3. 투명한 실행 과정:
   - 스트리밍을 통해 각 단계 관찰 가능
   - 디버깅과 개선이 용이함

4. 확장성:
   - 단일 도구에서 다중 도구로 쉽게 확장
   - 복잡한 워크플로우로 발전 가능

5. 사용자 경험:
   - "생각하는 과정"을 보여줌으로써 신뢰도 향상
   - 필요할 때만 외부 정보 조회 (불필요한 지연 방지)
"""

# =============================================
# 결론 및 향후 발전 방향:
# =============================================
"""
이번 실습을 통해 우리는:
1. LangGraph의 create_react_agent 함수로 복잡한 ReAct 에이전트를 간단히 생성
2. 에이전트가 상황에 맞게 도구 사용을 자동 결정하는 원리 이해
3. 실시간 웹 검색을 통한 최신 정보 획득 방법 습득
4. 스트리밍을 통한 에이전트 실행 과정 모니터링 방법 학습

향후 발전 방향:
1. 다중 도구 통합: 검색, 계산, API 호출 등 다양한 도구 추가
2. 메모리 시스템: 대화 기록을 장기적으로 기억하는 메모리 추가
3. 인간 피드백: 중요한 결정 시 사용자 확인 요청
4. 성능 최적화: 캐싱, 병렬 실행, 비동기 처리 등
5. 특화 에이전트: 금융, 의료, 교육 등 도메인 특화 에이전트 개발

이렇게 구축된 에이전트는 단순한 챗봇을 넘어서서
외부 세계와 상호작용하며 실제 문제를 해결하는
지능형 시스템의 기초가 될 수 있습니다.
"""


 질문: 사과는 영어로 뭐야?
--------------------------------------------------
================[답변] 사과는 영어로 "apple"이라고 합니다.
